	Heart Disease Prediction Using Machine Learning	Implement a Random Forest model from scratch without using scikit-learn, manually constructing multiple decision trees, aggregating predictions, and evaluating performance using custom accuracy calculations.

In [1]:
import pandas as pd
import random
import math

# Load training and testing datasets
train_df = pd.read_csv("heart_disease_train.csv")
test_df = pd.read_csv("heart_disease_test.csv")

feature_columns = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

target_column = "target"

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

Training rows: 79
Testing rows: 21


In [2]:
# Calculate Gini impurity
def gini_impurity(labels):

    if len(labels) == 0:
        return 0

    counts = {}

    for label in labels:

        if label not in counts:
            counts[label] = 0

        counts[label] += 1

    impurity = 1

    for label in counts:

        probability = counts[label] / len(labels)

        impurity -= probability ** 2

    return impurity


# Find majority class
def majority_class(labels):

    counts = {}

    for label in labels:

        if label not in counts:
            counts[label] = 0

        counts[label] += 1

    best_label = None
    best_count = -1

    for label in counts:

        if counts[label] > best_count:
            best_count = counts[label]
            best_label = label

    return best_label

In [3]:
# Find the best feature and threshold
def find_best_split(data, features):

    best_feature = None
    best_threshold = None
    best_gain = 0

    labels = list(data[target_column])

    parent_gini = gini_impurity(labels)

    for feature in features:

        values = []

        for value in data[feature]:

            if pd.notna(value):
                values.append(float(value))

        values.sort()

        thresholds = []

        for i in range(len(values) - 1):

            if values[i] != values[i + 1]:

                threshold = (
                    values[i] + values[i + 1]
                ) / 2

                thresholds.append(threshold)

        for threshold in thresholds:

            left_labels = []
            right_labels = []

            for i in range(len(data)):

                value = data[feature].iloc[i]
                label = data[target_column].iloc[i]

                if pd.isna(value) or pd.isna(label):
                    continue

                if float(value) <= threshold:
                    left_labels.append(label)
                else:
                    right_labels.append(label)

            if not left_labels or not right_labels:
                continue

            left_gini = gini_impurity(left_labels)
            right_gini = gini_impurity(right_labels)

            total = len(left_labels) + len(right_labels)

            weighted_gini = (
                len(left_labels) / total * left_gini
                + len(right_labels) / total * right_gini
            )

            gain = parent_gini - weighted_gini

            if gain > best_gain:

                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold, best_gain

In [4]:
# Build one decision tree
def build_tree(data, features, depth=0, max_depth=5, min_samples=5):

    labels = list(data[target_column])

    if len(labels) == 0:
        return {"leaf": True, "prediction": 0}

    # Stop if only one class remains
    unique_labels = []

    for label in labels:

        if label not in unique_labels:
            unique_labels.append(label)

    if len(unique_labels) == 1:

        return {
            "leaf": True,
            "prediction": unique_labels[0]
        }

    # Stop at maximum depth or minimum sample count
    if depth >= max_depth or len(data) < min_samples:

        return {
            "leaf": True,
            "prediction": majority_class(labels)
        }

    # Randomly select features for this node
    number_to_select = max(
        1,
        int(math.sqrt(len(features)))
    )

    selected_features = random.sample(
        features,
        min(number_to_select, len(features))
    )

    feature, threshold, gain = find_best_split(
        data,
        selected_features
    )

    if feature is None:

        return {
            "leaf": True,
            "prediction": majority_class(labels)
        }

    left_rows = []
    right_rows = []

    for i in range(len(data)):

        value = data[feature].iloc[i]

        if pd.isna(value):
            continue

        if float(value) <= threshold:
            left_rows.append(i)
        else:
            right_rows.append(i)

    if not left_rows or not right_rows:

        return {
            "leaf": True,
            "prediction": majority_class(labels)
        }

    left_data = data.iloc[left_rows].reset_index(drop=True)
    right_data = data.iloc[right_rows].reset_index(drop=True)

    return {
        "leaf": False,
        "feature": feature,
        "threshold": threshold,
        "left": build_tree(
            left_data,
            features,
            depth + 1,
            max_depth,
            min_samples
        ),
        "right": build_tree(
            right_data,
            features,
            depth + 1,
            max_depth,
            min_samples
        )
    }

In [5]:
# Predict one observation
def predict_tree(tree, row):

    if tree["leaf"]:
        return tree["prediction"]

    feature = tree["feature"]
    threshold = tree["threshold"]

    value = row[feature]

    if pd.isna(value):
        return tree["prediction"] if "prediction" in tree else 0

    if float(value) <= threshold:
        return predict_tree(tree["left"], row)

    return predict_tree(tree["right"], row)

In [6]:
# Predict one observation
def predict_tree(tree, row):

    if tree["leaf"]:
        return tree["prediction"]

    feature = tree["feature"]
    threshold = tree["threshold"]

    value = row[feature]

    if pd.isna(value):
        return predict_tree(tree["left"], row)

    if float(value) <= threshold:
        return predict_tree(tree["left"], row)

    return predict_tree(tree["right"], row)

In [7]:
# Configure Random Forest
random.seed(42)

number_of_trees = 15
forest = []

# Remove rows with missing required values
clean_train = train_df[
    feature_columns + [target_column]
].dropna().reset_index(drop=True)

# Train multiple trees using bootstrap samples
for tree_number in range(number_of_trees):

    bootstrap_rows = []

    for i in range(len(clean_train)):

        random_index = random.randint(
            0,
            len(clean_train) - 1
        )

        bootstrap_rows.append(random_index)

    bootstrap_data = clean_train.iloc[
        bootstrap_rows
    ].reset_index(drop=True)

    tree = build_tree(
        bootstrap_data,
        feature_columns,
        max_depth=5,
        min_samples=5
    )

    forest.append(tree)

    print(
        "Tree",
        tree_number + 1,
        "trained"
    )

print("\nRandom Forest training completed!")
print("Total trees:", len(forest))

Tree 1 trained
Tree 2 trained
Tree 3 trained
Tree 4 trained
Tree 5 trained
Tree 6 trained
Tree 7 trained
Tree 8 trained
Tree 9 trained
Tree 10 trained
Tree 11 trained
Tree 12 trained
Tree 13 trained
Tree 14 trained
Tree 15 trained

Random Forest training completed!
Total trees: 15


In [8]:
# Predict using all trees
def predict_forest(forest, row):

    votes = {}

    for tree in forest:

        prediction = predict_tree(tree, row)

        if prediction not in votes:
            votes[prediction] = 0

        votes[prediction] += 1

    return majority_class_from_votes(votes)


# Find the class with the most votes
def majority_class_from_votes(votes):

    best_class = None
    highest_votes = -1

    for label in votes:

        if votes[label] > highest_votes:
            highest_votes = votes[label]
            best_class = label

    return best_class


# Generate test predictions
clean_test = test_df[
    feature_columns + [target_column]
].dropna().reset_index(drop=True)

actual_values = []
predicted_values = []

for i in range(len(clean_test)):

    row = clean_test.iloc[i]

    actual_values.append(
        row[target_column]
    )

    prediction = predict_forest(
        forest,
        row
    )

    predicted_values.append(prediction)

print("Predictions generated:", len(predicted_values))

Predictions generated: 21


In [9]:
# Calculate accuracy
correct_predictions = 0

for i in range(len(actual_values)):

    if actual_values[i] == predicted_values[i]:
        correct_predictions += 1

if len(actual_values) > 0:

    accuracy = (
        correct_predictions
        / len(actual_values)
    ) * 100

else:
    accuracy = 0


print("\nRANDOM FOREST EVALUATION")
print("=" * 40)

print("Correct predictions:", correct_predictions)
print("Total predictions:", len(actual_values))

print(
    "Accuracy:",
    round(accuracy, 2),
    "%"
)


RANDOM FOREST EVALUATION
Correct predictions: 16
Total predictions: 21
Accuracy: 76.19 %


In [10]:
# Display actual and predicted results
results = pd.DataFrame({
    "Actual": actual_values,
    "Predicted": predicted_values
})

print(results.head(15))

print("\nPrediction comparison completed.")

    Actual  Predicted
0      1.0          1
1      1.0          1
2      1.0          1
3      1.0          0
4      1.0          1
5      1.0          1
6      1.0          0
7      1.0          1
8      1.0          1
9      0.0          0
10     1.0          1
11     0.0          1
12     1.0          1
13     1.0          1
14     1.0          1

Prediction comparison completed.
